# ProGen2-XL scoring of the experimental-PDB cohort

Scores the 876 PDB chains in `cohort_pdb_scoring_inputs.csv` with the
autoregressive protein language model ProGen2-XL (Nijkamp et al., 2023). This
produces the `progen2_XL_score` column of `data/pdb/cohort_pdb_scored.csv`.
Scoring uses the `sequence` field only; no structure is needed.

Scoring convention:
- For each sequence `s = s_1 ... s_L`, the total log-likelihood
  `sum_i log P(s_i | s_{<i})` comes from one teacher-forced forward pass.
- `progen2_XL_score` is the per-residue mean log-likelihood (`total / L`), on
  the same scale as the other per-residue scores; higher (closer to 0) means
  the model assigns more probability to the sequence.
- The total log-likelihood and token count are also written.

Sequence cleaning:
- 20 standard amino acids only.
- A terminal `*` is stripped; X / B / Z / U / O are not replaced.
- Sequences with non-standard residues get NaN scores and
  `sequence_filter_status="nonstandard_amino_acid"`.

Checkpoint: `hugohrban/progen2-xlarge`. Set `PROGEN2_CHECKPOINT` in the
configuration cell to `hugohrban/progen2-base` for the base checkpoint.

## 1. Setup

In [ ]:
!nvidia-smi -L


In [ ]:
# transformers 4.x loads the ProGen2 checkpoints on the Hugging Face Hub.
!pip install -q 'transformers>=4.40,<5' 'tokenizers>=0.15' 'accelerate>=0.30' biopython tqdm


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DATASET = '/content/cohort_pdb_scoring_inputs.csv'

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding-design-bias/outputs'
OUTPUT        = f'{DRIVE_OUT_DIR}/progen2_XL_scores_pdb.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

PROGEN2_CHECKPOINT = 'hugohrban/progen2-xlarge'

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)
print('checkpoint ->', PROGEN2_CHECKPOINT)


## 2. Load model

In [ ]:
import torch
import warnings; warnings.filterwarnings('ignore')
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

tokenizer = AutoTokenizer.from_pretrained(PROGEN2_CHECKPOINT, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    PROGEN2_CHECKPOINT,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).to(device).eval()

# ProGen2 uses '1' as BOS in the published code. Verify from the loaded tokeniser
# and fall back to '1' with a warning if the tokeniser doesn't expose it.
BOS_ID = tokenizer.bos_token_id
if BOS_ID is None:
    BOS_ID = 1
    print(f'tokenizer.bos_token_id is None; falling back to BOS_ID={BOS_ID} '
          f'(ProGen2 paper / Nijkamp code uses 1).')
else:
    print(f'BOS_ID from tokenizer: {BOS_ID}')

print(f'model dtype: {next(model.parameters()).dtype}')
print(f'vocab size:  {tokenizer.vocab_size}')


## 3. Sequence cleaning + scoring wrapper

In [ ]:
import pandas as pd
import math

VALID_STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

def clean_sequence(seq):
    """Return the cleaned sequence and any non-standard characters it contains."""
    if seq is None or (isinstance(seq, float) and math.isnan(seq)):
        return '', 'missing'
    s = str(seq).strip().upper().replace(' ', '').replace('\n', '').replace('\r', '')
    if s.endswith('*'):
        s = s[:-1]
    bad_chars = ''.join(sorted(set(s) - VALID_STANDARD_AA))
    return s, bad_chars


def classify(seq_clean, bad_chars):
    if not seq_clean:
        return 'empty_sequence'
    if bad_chars:
        return 'nonstandard_amino_acid'
    return 'included'


@torch.no_grad()
def score_sequence_progen2(seq):
    """Teacher-forced total + mean log-likelihood for one WT sequence.

    Input tokens:   [BOS, t_1, ..., t_L]
    Forward pass:   logits[i] predicts t_{i+1} given t_{<=i}
    Score sum:      sum_{i=0..L-1} log p(t_{i+1} | t_{<=i})

    Returns a dict with
      - progen2_XL_total_logp : sum of log p over all L positions
      - progen2_XL_score      : mean per-residue log p (= total / L)
      - scored_length         : L (number of residues scored)
      - num_tokens            : L (ProGen2 has one token per residue)
    """
    enc = tokenizer(seq, add_special_tokens=False, return_tensors='pt')
    token_ids = enc['input_ids'][0]
    if BOS_ID is not None:
        token_ids = torch.cat([torch.tensor([BOS_ID]), token_ids])
    token_ids = token_ids.to(device)

    # Single forward pass; shift logits/labels to align next-token predictions.
    out = model(input_ids=token_ids.unsqueeze(0))
    logits = out.logits[0]                  # (T, V)
    log_probs = torch.log_softmax(logits.float(), dim=-1)

    # Predict token at index i+1 from logits at index i; labels are token_ids[1:]
    next_tokens = token_ids[1:]             # (T-1,)
    pred_log_probs = log_probs[:-1]         # (T-1, V)
    picked = pred_log_probs[torch.arange(next_tokens.shape[0], device=device), next_tokens]

    total_logp = float(picked.sum().item())
    L = int(next_tokens.shape[0])
    return {
        'progen2_XL_total_logp': total_logp,
        'progen2_XL_score':      total_logp / L if L else float('nan'),
        'scored_length':      L,
        'num_tokens':         L,
    }


def score_with_oom_retry(seq):
    """Score one sequence, recording an out-of-memory error instead of stopping the run."""
    try:
        return score_sequence_progen2(seq), None
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        return None, f'out_of_memory: L={len(seq)}'
    except Exception as exc:
        return None, repr(exc)


## 4. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    seq, bad = clean_sequence(r.get('sequence', ''))
    status = classify(seq, bad)
    if status != 'included':
        print(entry, status, bad); continue
    t0 = time.time(); res, err = score_with_oom_retry(seq); t = time.time() - t0
    if res is None:
        print(entry, 'failed:', err); continue
    print(f'{entry} L={res["scored_length"]:4d} '
          f'progen2_XL_score={res["progen2_XL_score"]:.4f}  total={res["progen2_XL_total_logp"]:.2f}  '
          f'({t:.1f}s)')


## 5. Full run with resume

In [ ]:
from tqdm.auto import tqdm

# Writes a fresh OUTPUT file. Shortest sequences first, so any out-of-memory
# failures come last.
todo = sorted(rows, key=lambda r: len(r.get('sequence', '')))
print('to score:', len(todo))

FIELDS = [
    'Entry', 'species', 'domain',
    'progen2_XL_score', 'progen2_XL_total_logp',
    'scored_length', 'num_tokens',
    'dataset_length', 'sequence_filter_status', 'error',
]

with open(OUTPUT, 'w', newline='') as out:
    w = csv.DictWriter(out, fieldnames=FIELDS)
    w.writeheader()

    counts = {'ok': 0, 'skipped_nonstd': 0, 'oom': 0, 'error': 0, 'empty': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='progen2'):
        entry = r['Entry']
        rec = {
            'Entry': entry,
            'species': r.get('species', ''),
            'domain':  r.get('domain', ''),
            'progen2_XL_score': '',
            'progen2_XL_total_logp': '',
            'scored_length': 0,
            'num_tokens': 0,
            'dataset_length': len(r.get('sequence', '')),
            'sequence_filter_status': '',
            'error': '',
        }
        seq, bad = clean_sequence(r.get('sequence', ''))
        status = classify(seq, bad)
        rec['sequence_filter_status'] = status
        if status != 'included':
            if status == 'empty_sequence':
                counts['empty'] += 1
            else:
                counts['skipped_nonstd'] += 1
                rec['error'] = f'bad_chars={bad}'
            w.writerow(rec); out.flush(); continue

        res, err = score_with_oom_retry(seq)
        if res is None:
            rec['error'] = err or 'unknown_error'
            if err and 'out_of_memory' in err:
                counts['oom'] += 1
            else:
                counts['error'] += 1
            w.writerow(rec); out.flush(); continue

        rec.update(res)
        w.writerow(rec); out.flush()
        counts['ok'] += 1
        if counts['ok'] % 50 == 0:
            torch.cuda.empty_cache()

print('done in', round(time.time() - t_start, 1), 's', counts)


## 6. Summary

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
print('non-null progen2_XL_score:', df['progen2_XL_score'].notna().sum())
df[['progen2_XL_score', 'progen2_XL_total_logp', 'scored_length']].describe()